### ***Model T5 Fine Tuning***

In [1]:
import pandas as pd
import numpy as np

train_data = pd.read_csv("../content/samsum-train.csv", engine='python', on_bad_lines='warn')
val_data = pd.read_csv("../content/samsum-validation.csv", engine='python', on_bad_lines='warn')

In [2]:
train_data.sample(7)

,id,dialogue,summary
263,13680964,"Devlin: u have new bad?\r\nKira: yes, I bought...",Kira bought a new bed this weekend at a new sh...
2678,13728743,Amit: How many foamed cabinets are left in the...,"There are 3,432 foamed cabinets left in the fi..."
11167,13716306,Elina: You are 6 months already? Omg!! 😃\r\nHa...,Hannah is 6 months pregnant and stays fit.
3058,13811847,"Gwen: Great job, Joe, you only had one job - w...",Joe didn't water the plants and killed them.
7648,13728652-1,"Ben: Hi , Nathan is expected whenever he wants...",Clarisse enjoyed their visit to Blainville. Be...
5280,13818752,Austin: <file_photo> First attempt with the sl...,"Austin made a dish with a slow cooker, but she..."
4415,13812748,"Nicole: Hi Ruth, how are you?\r\nRuth: I’m fin...",Ruth was dismissed from W&A by Jack. Jack was ...


In [3]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [4]:
train_data.shape

(13168, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
# apply random sampling
train_data = train_data.sample(n=10000,random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [7]:
train_data.shape

(10000, 3)

#### ***Data Pre-Processing***

In [8]:
import re

# clean our dataset
def clean_data(text):
   # Ensure the input is a string before applying regex
   text = str(text)
   text = re.sub(r"\r\n", " ", text) # lines
   text = re.sub(r"\s+", " ", text)  # spaces
   text = re.sub(r"<.*?>", " ", text)
   text = text.strip().lower()
   return text

# Training Data
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

# Validation Data
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

#### ***Tokenization***

In [9]:
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# raw data --> fine tunning
def tokenize(data):
  inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
  targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

  # add token id -> input labels
  inputs["labels"] = targets["input_ids"]
  return inputs

train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [10]:
train_dataset[0]

{'input_ids': [3, 29, 9, 6736, 10, 3, 23, 31, 51, 773, 9, 19682, 3, 29, 9, 6736, 10, 103, 25, 43, 136, 126, 21705, 24, 25, 54, 1568, 58, 2662, 1050, 10, 6865, 3, 10, 61, 2662, 1050, 10, 131, 5607, 140, 125, 773, 13, 21705, 33, 25, 1638, 16, 58, 3, 29, 9, 6736, 10, 424, 659, 11, 6613, 3, 29, 9, 6736, 10, 1066, 4092, 42, 11043, 3803, 3, 29, 9, 6736, 10, 59, 3, 9, 600, 1819, 13, 17201, 18, 89, 23, 2662, 1050, 10, 410, 25, 1605, 959, 45, 48, 1590, 87, 210, 3870, 5818, 58, 3, 29, 9, 6736, 10, 59, 780, 2662, 1050, 10, 207, 6, 24, 3231, 178, 28, 128, 1245, 931, 2662, 1050, 10, 166, 13, 66, 1077, 82, 1305, 126, 1764, 96, 6279, 97, 3, 23, 530, 3, 60, 18860, 920, 38, 3, 9, 12593, 15, 121, 2662, 1050, 10, 8, 2233, 845, 66, 81, 34, 3, 29, 9, 6736, 10, 24, 31, 7, 3, 9, 17056, 564, 2662, 1050, 10, 168, 17945, 68, 8, 21705, 19, 248, 2662, 1050, 10, 34, 65, 128, 19752, 8073, 68, 167, 13, 8, 97, 34, 11331, 7, 66, 13, 39, 5598, 2662, 1050, 10, 659, 6, 6613, 11, 11043, 1898, 3, 29, 9, 6736, 10, 2993, 147

In [11]:
len(train_dataset[0]["input_ids"])

512

### ***Working With Our Model***

In [12]:
from transformers import T5ForConditionalGeneration
# NLP  --> Genrate Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
# Training Arguments
from transformers import TrainingArguments,Trainer

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500,
    optim="adamw_torch",
    torch_compile=False
    # 0 => lr default
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [18]:
trainer.train() # now train our model

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.405538,0.354459
2,0.372604,0.344296
3,0.359575,0.338724
4,0.349522,0.336687
5,0.344559,0.335722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.6695534313964844, metrics={'train_runtime': 1482.1444, 'train_samples_per_second': 33.735, 'train_steps_per_second': 4.217, 'total_flos': 6767090073600000.0, 'train_loss': 0.6695534313964844, 'epoch': 5.0})

#### ***Save Our Model***

In [21]:
import os
import torch

save_directory = "./summary_model"
os.makedirs(save_directory, exist_ok=True)

# Save the model state_dict directly using torch.save
torch.save(model.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))
tokenizer.save_pretrained(save_directory)

('./summary_model/tokenizer_config.json', './summary_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./summary_model")
tokenizer = T5Tokenizer.from_pretrained("./summary_model")

In [22]:
## Test Our Model
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [23]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.
Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.
Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.
Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.
Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.
Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.
Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.
Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')